# 02 · Análisis exploratorio unificado — Éxito de videojuegos en Steam

**Proyecto final · Aprendizaje de Máquina (ITAM, otoño 2026)** · Notebook del equipo (Filiberto, Diana, Alejandra y Luis)

Este notebook unifica los cuatro EDA individuales (Filiberto: planteamiento y diccionario; Diana; Luis; Alejandra) en un solo análisis, con una única versión de datos y las decisiones de la sesión del 22 de noviembre. Es la base de la sección de EDA del reporte (`docs/03-analisis-exploratorio.md`) y fija los parámetros que usa el pipeline (`src/steam_exito/config.py`).

**Pregunta:** ¿qué factores, disponibles *antes* del lanzamiento, se asocian con una mayor proporción de reseñas positivas en un juego nuevo de Steam?

**Decisiones vigentes que se aplican aquí**

| ID | Decisión |
|---|---|
| D-002 | Se excluyen los juegos lanzados en 2025 y 2026 |
| D-003 | Variable objetivo: proporción de reseñas positivas |
| D-004 | Se elimina el registro con precio atípico (107,500 USD) |
| D-005 | Las etiquetas de marketing (Indie, Early Access, Free to Play) se separan de los géneros; se quitan los no-juegos |
| D-006 / D-009 | No se usa el nombre del desarrollador; se derivan variables de su historial (y del publisher) |
| D-007 | Sin procesamiento de lenguaje natural |
| D-008 | Sin ponderar por número de reseñas: la muestra se define por umbral y por año |
| D-010 *(propuesta)* | Las variables Pre\* (valor actual de la tienda, no el del lanzamiento) entran con reserva y se auditan |

**Convención:** `pct_pos = positive / (positive + negative)`. Para comparar entre épocas se usa también `dif_cohorte = pct_pos − media de su año de lanzamiento` (positivo = mejor que el promedio de su año).

In [ ]:
import sys
import warnings
from pathlib import Path

RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from scipy.stats import spearmanr

from steam_exito import config
from steam_exito.carga import cargarSlim, verificarVersion
from steam_exito.parsers import aLista, idiomasYAudio
from steam_exito.graficas import estiloBase, guardarFigura, guardarMetrica, COLOR, COLOR_2

warnings.filterwarnings("ignore")
estiloBase()
pd.set_option("display.max_columns", 40, "display.width", 150, "display.float_format", "{:,.3f}".format)
np.random.seed(config.SEMILLA)

UMBRAL, ANIO_MAX = config.UMBRAL_RESENAS, config.ANIO_MAX
R = {}  # cifras clave que alimentan la tabla final de hallazgos


def rho(a, b):
    """Correlación de Spearman ignorando faltantes."""
    ok = a.notna() & b.notna()
    return spearmanr(a[ok], b[ok])[0] if ok.sum() > 10 else np.nan


raw = cargarSlim()
verificarVersion(raw, estricto=False)  # avisa si no es la versión congelada (A-005)
print(f"Versión congelada: descarga {config.VERSION_DATOS['fecha_descarga']} · {raw.shape[0]:,} filas × {raw.shape[1]} columnas")

## 1. Fuente y estructura de los datos

**Fuente:** *Steam Games Dataset – All Games* (Hubert Sidorowicz, Kaggle, licencia MIT), con datos de Steam Store y estimaciones de SteamSpy. Se actualiza a diario, por eso el equipo trabaja con **una sola versión congelada** (descarga del 24 de noviembre de 2026). Los cuatro EDA individuales se hicieron con descargas distintas (145,257; 150,199; 150,643 y 151,023 filas) y por eso sus cifras no coincidían.

Se usa el archivo `steam_games.csv` (unidad de análisis: un juego, llave `app_id`). Se retiraron del archivo las columnas de texto pesado (descripciones, imágenes, capturas, tráileres, paquetes, etiquetas de usuarios) y se conservaron indicadores `has_*` para sitio web, soporte y aviso de contenido. El archivo `steam_games_reviews.csv` solo contiene citas de prensa del 9 % de los juegos: no son reseñas de usuarios y no se usa.

In [ ]:
momento = {"Post (se genera después del lanzamiento)": [c for c in raw.columns if c in config.COLUMNAS_POST],
           "Pre* (valor actual de la tienda, puede diferir del del lanzamiento)": [c for c in raw.columns if c in config.VARIABLES_PRE_ASTERISCO]}
resto = [c for c in raw.columns if c not in sum(momento.values(), [])]
momento["Pre / identificadores / indicadores"] = resto
for k, v in momento.items():
    print(f"{k} ({len(v)}):\n   {', '.join(v)}\n")
print("tipos:", raw.dtypes.astype(str).value_counts().to_dict(), "| app_id único:", raw.app_id.is_unique)

## 2. Calidad de los datos

### 2.1 Faltantes y valores que significan «sin dato»

En este conjunto un faltante puede venir de cuatro formas: nulo, cadena vacía o lista `[]`, y **cero** (en nueve columnas la fuente usa 0 o el rango `0 - 0` cuando no tiene información). Medir solo los nulos subestima el problema; aquí se cuentan las cuatro.

In [ ]:
def perfilCalidad(d):
    filas = []
    for c in d.columns:
        s = d[c]
        texto = s.astype("string").str.strip()
        nulos = int(s.isna().sum())
        vacios = int(texto.isin(["", "[]", "{}"]).sum())
        numerica = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
        ceros = int((s == 0).sum()) if numerica else 0
        if c == "estimated_owners":
            ceros = int(texto.isin(["0 - 0", "0 .. 0"]).sum())
        cero_nulo = c in config.COLUMNAS_CERO_ES_NULO or c == "estimated_owners"
        sin_dato = nulos + vacios + (ceros if cero_nulo else 0)
        filas.append({"columna": c, "nulos": nulos, "vacios_o_lista_vacia": vacios, "ceros": ceros,
                      "el_cero_es_sin_dato": cero_nulo, "pct_sin_dato": 100 * sin_dato / len(d)})
    return pd.DataFrame(filas).set_index("columna").sort_values("pct_sin_dato", ascending=False)


calidad = perfilCalidad(raw)
guardarMetrica(calidad, "02_eda_calidad_columnas")
display(calidad.head(16).round(2))

top = calidad.head(14).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(top.index, top["pct_sin_dato"], color=np.where(top["el_cero_es_sin_dato"], COLOR_2, COLOR))
ax.set_xlabel("% de juegos sin dato (nulo, vacío, lista vacía o 0 que significa «sin dato»)")
ax.set_title("Figura 1. Columnas con más información faltante (naranja: el 0 es «sin dato»)")
guardarFigura(fig, "02_eda_01_faltantes"); plt.show()

**Lectura.** Las columnas de actividad posterior al lanzamiento (`peak_ccu`, tiempos de juego, `metacritic_score`, `recommendations`, `user_score`) están vacías en la mayoría de los juegos: son escasas **y** posteriores al lanzamiento, así que no sirven como predictoras (solo para validar). Los faltantes de `developers`, `publishers`, `genres`, `categories` e idiomas (~6 %) coinciden en las mismas filas. En `full_audio_languages` el `[]` es un valor real («sin doblaje»), no un faltante.

### 2.2 Filas que se eliminan: fechas, precios y no-juegos

Reglas ya acordadas: fechas inválidas (82), precio atípico (D-004) y títulos que no son videojuegos (software, video; D-005). El filtro de no-juegos se aplica sobre los géneros ya normalizados (se mapean las variantes traducidas y se ignoran las etiquetas de marketing y de aviso al decidir).

In [ ]:
df = raw.copy()
for c in ["positive", "negative", "price"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["fecha"] = pd.to_datetime(df["release_date"], errors="coerce")
df["anio"] = df["fecha"].dt.year
df["n_rev"] = df["positive"] + df["negative"]
df["pct_pos"] = df["positive"] / df["n_rev"].replace(0, np.nan)

# Listas normalizadas (un solo parser para los formatos del dataset)
df["lista_gen"] = df["genres"].map(aLista).map(lambda L: [config.MAPA_GENEROS_TRADUCIDOS.get(g, g) for g in L])
df["lista_cat"] = df["categories"].map(aLista)
df["lista_dev"] = df["developers"].map(aLista)
df["lista_pub"] = df["publishers"].map(aLista)
_pares = [idiomasYAudio(s, a) for s, a in zip(df["supported_languages"], df["full_audio_languages"])]
df["lista_idi"] = [p[0] for p in _pares]
df["lista_audio"] = [p[1] for p in _pares]   # None = no se sabe (formato texto sin full_audio_languages)

ignorar = set(config.ETIQUETAS_MARKETING) | set(config.AVISO_CONTENIDO)
def esNoJuego(L):
    nucleo = [g for g in L if g not in ignorar]
    return len(nucleo) > 0 and all(g in config.GENEROS_NO_JUEGO for g in nucleo)
df["no_juego"] = df["lista_gen"].map(esNoJuego)

limpieza = pd.DataFrame([
    ("Archivo congelado", len(df)),
    ("− fecha de lanzamiento inválida", int(df.fecha.isna().sum())),
    ("− precio atípico (D-004)", int((df.price > config.PRECIO_MAX_VALIDO).sum())),
    ("− no son juegos (D-005)", int(df.no_juego.sum())),
], columns=["paso", "juegos"]).set_index("paso")

u = df[df.fecha.notna() & ~(df.price > config.PRECIO_MAX_VALIDO) & ~df.no_juego].copy()   # universo limpio
u["anio"] = u["anio"].astype(int)
u_ok = u[u.anio <= ANIO_MAX].copy()                                                          # hasta 2024 (D-002)
m = u_ok[u_ok.n_rev >= UMBRAL].copy()                                                        # muestra de modelado
limpieza.loc["= universo limpio (todos los años)"] = len(u)
limpieza.loc[f"− lanzados después de {ANIO_MAX} (D-002)"] = len(u) - len(u_ok)
limpieza.loc[f"= universo hasta {ANIO_MAX}"] = len(u_ok)
limpieza.loc[f"− con menos de {UMBRAL} reseñas"] = len(u_ok) - len(m)
limpieza.loc["= muestra de modelado"] = len(m)
guardarMetrica(limpieza, "02_eda_embudo_filas")
display(limpieza)
print("Precios mayores a 1,000 USD (todos son de 2025-2026, por lo que D-002 también los excluye):")
print(df.loc[df.price > 1000, ["name", "price", "anio", "n_rev"]].sort_values("price").to_string(index=False))
R.update(n_raw=len(df), n_fechas=int(df.fecha.isna().sum()), n_nojuego=int(df.no_juego.sum()), n_u=len(u), n_uok=len(u_ok), n_m=len(m))
print("Duplicados de app_id:", int(raw.app_id.duplicated().sum()))

### 2.3 Formato mixto de `genres` y `categories`

Las listas vienen en dos formatos (lista de textos u objetos con `description`), y el cambio ocurre alrededor de 2019–2020. Es un artefacto de la recolección, **no una característica del juego**, pero coincide con una diferencia de recepción por época: si se usara como variable, el modelo aprendería el año de lanzamiento.

In [ ]:
df["fmt_dict"] = raw["genres"].str.contains('"description"', na=False)
por_anio = df[df.anio.between(2012, 2026)].groupby("anio")["fmt_dict"].mean()
mm = df.loc[m.index].assign(fmt=lambda d: np.where(d.fmt_dict, "objeto", "texto"))
mm["dif"] = m.pct_pos - m.groupby("anio").pct_pos.transform("mean")
global_fmt = mm.groupby("fmt").pct_pos.agg(["size", "mean"])
dentro_cohorte = mm.groupby("fmt").dif.mean()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(por_anio.index, 100 * por_anio.values, marker="o", color=COLOR)
ax[0].set(title="Figura 2a. % de juegos con `genres` en formato objeto", xlabel="año de lanzamiento", ylabel="%")
ax[1].bar(["objeto", "texto"], [mm[mm.fmt == "objeto"].pct_pos.mean(), mm[mm.fmt == "texto"].pct_pos.mean()], color=[COLOR, COLOR_2])
ax[1].set(title="Figura 2b. pct_pos medio por formato (misma muestra)", ylim=(0.7, 0.85))
guardarFigura(fig, "02_eda_02_formato_mixto"); plt.show()
print("pct_pos medio por formato:\n", global_fmt.round(3).to_string())
print("\nDiferencia dentro de cada año (media de dif_cohorte):", dentro_cohorte.round(3).to_dict())
R["fmt_dif_global"] = float(global_fmt["mean"].diff().iloc[-1])
R["fmt_dif_cohorte"] = float(dentro_cohorte.diff().iloc[-1])

**Conclusión.** La diferencia global entre formatos desaparece casi por completo al comparar dentro del mismo año. Decisión: leer ambos formatos con `parsers.aLista` y **no usar el formato como variable**.

## 3. Variable objetivo y definición de la muestra

### 3.1 ¿Cuántos juegos tienen reseñas y qué umbral conviene?

Un `pct_pos` calculado con pocas reseñas es ruidoso (1 de 1 = 100 %). El umbral se elige viendo cuántos juegos sobreviven y cuánto baja la dispersión. La tabla compara todos los años con la muestra hasta 2024 (D-002).

In [ ]:
sin_resenas = int((u_ok.n_rev == 0).sum())
R["sin_resenas_pct"] = 100 * sin_resenas / len(u_ok)
print(f"Juegos sin ninguna reseña (hasta {ANIO_MAX}, universo limpio): {sin_resenas:,} ({R['sin_resenas_pct']:.1f} %)")

filas = []
for t in [1, 10, 30, 50, 100, 500]:
    s = u_ok[u_ok.n_rev >= t]
    filas.append({"umbral": t, "juegos_todos_los_anios": int((u.n_rev >= t).sum()), f"juegos_hasta_{ANIO_MAX}": len(s),
                  "pct_del_universo": 100 * len(s) / len(u_ok), "media_pct_pos": s.pct_pos.mean(), "desv_pct_pos": s.pct_pos.std()})
umbrales = pd.DataFrame(filas).set_index("umbral")
guardarMetrica(umbrales, "02_eda_umbrales")
display(umbrales.round(3))
R["std_t1"], R["std_t50"] = umbrales.loc[1, "desv_pct_pos"], umbrales.loc[50, "desv_pct_pos"]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(m.pct_pos, bins=40, color=COLOR)
ax[0].axvline(m.pct_pos.median(), color="k", ls="--", lw=1)
ax[0].set(title=f"Figura 3a. pct_pos (≥{UMBRAL} reseñas, n={len(m):,})", xlabel="proporción positiva")
ax[1].hist(np.log10(u_ok[u_ok.n_rev > 0].n_rev), bins=40, color=COLOR_2)
ax[1].axvline(np.log10(UMBRAL), color="k", ls="--", lw=1)
ax[1].set(title="Figura 3b. Reseñas por juego (log10, ≥1)", xlabel="log10(# reseñas)")
ax[2].plot(umbrales.index.astype(str), umbrales["desv_pct_pos"], marker="o", color=COLOR)
ax[2].set(title="Figura 3c. Dispersión de pct_pos según el umbral", xlabel="umbral mínimo de reseñas", ylabel="desv. estándar")
guardarFigura(fig, "02_eda_03_objetivo_umbral"); plt.show()
print(f"Muestra (≥{UMBRAL}): media {m.pct_pos.mean():.3f} · mediana {m.pct_pos.median():.3f} · desv {m.pct_pos.std():.3f} · asimetría {m.pct_pos.skew():.2f} · "
      f"p5 {m.pct_pos.quantile(.05):.2f} · p95 {m.pct_pos.quantile(.95):.2f}")
R.update(media=m.pct_pos.mean(), mediana=m.pct_pos.median(), std=m.pct_pos.std(), asim=m.pct_pos.skew())

**Lectura.** La dispersión cae con rapidez hasta unas 50 reseñas y mejora poco después, mientras que el umbral de 100 quita una parte importante de la muestra. Se mantiene **50 como valor principal**, con sensibilidad a 30 y 100 (A-003). La distribución es asimétrica a la izquierda y se concentra entre 0.7 y 0.95: conviene reportar MAE además de RMSE.

### 3.2 Tres definiciones posibles de «éxito» (decisión 0001 abierta)

El equipo todavía no decide entre la proporción cruda, una versión que penaliza pocas reseñas (límite inferior de Wilson) y una versión relativa a la cohorte (percentil dentro del año de lanzamiento, o «cuartil superior» como etiqueta binaria). Se calculan las tres para decidir con evidencia.

In [ ]:
def wilsonInf(pos, n, z=1.96):
    p = pos / n
    return (p + z * z / (2 * n) - z * np.sqrt((p * (1 - p) + z * z / (4 * n)) / n)) / (1 + z * z / n)

m["wilson"] = wilsonInf(m.positive, m.n_rev)
m["pct_cohorte"] = m.groupby("anio").pct_pos.rank(pct=True)
m["bueno"] = (m.pct_cohorte >= 0.75).astype(int)
m["dif_cohorte"] = m.pct_pos - m.groupby("anio").pct_pos.transform("mean")

var = m[["pct_pos", "wilson", "pct_cohorte", "dif_cohorte"]]
corr_var = var.corr("spearman")
display(corr_var.round(3))
anual_v = m.groupby("anio")[["pct_pos", "wilson", "pct_cohorte"]].mean().loc[2012:]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for col, c, t in [("pct_pos", COLOR, "pct_pos"), ("wilson", COLOR_2, "límite inferior de Wilson")]:
    ax[0].hist(m[col], bins=40, alpha=.6, color=c, label=t)
ax[0].legend(); ax[0].set(title="Figura 4a. pct_pos vs Wilson", xlabel="valor")
ax[1].hist(m.pct_cohorte, bins=20, color=COLOR); ax[1].set(title="Figura 4b. Percentil dentro de la cohorte", xlabel="percentil")
for col, c in [("pct_pos", COLOR), ("wilson", COLOR_2), ("pct_cohorte", "gray")]:
    ax[2].plot(anual_v.index, anual_v[col], marker="o", color=c, label=col)
ax[2].legend(); ax[2].set(title="Figura 4c. Media anual de cada variante", xlabel="año de lanzamiento")
guardarFigura(fig, "02_eda_04_objetivo_variantes"); plt.show()
print(f"Etiqueta «bueno» (cuartil superior de su cohorte): {m.bueno.mean():.1%} de la muestra")
R["rho_wilson"] = corr_var.loc["pct_pos", "wilson"]

**Lectura.** `pct_pos` y su límite de Wilson casi coinciden con ≥50 reseñas (Spearman cercano a 1): la corrección de Wilson solo importaría con umbrales bajos. El percentil por cohorte elimina por construcción la tendencia anual (Figura 4c). **Propuesta:** `pct_pos` crudo como objetivo principal de regresión (más simple y es lo acordado en D-003), y la etiqueta de cuartil superior por cohorte como objetivo de clasificación secundario; ambos salen del pipeline (`y_*`).

### 3.3 Qué juegos entran a la muestra: sesgo de selección y de antigüedad

In [ ]:
por = u[u.anio.between(2010, 2026)].groupby("anio").agg(juegos=("app_id", "size"), con_umbral=("n_rev", lambda s: int((s >= UMBRAL).sum())))
por["pct_con_umbral"] = 100 * por.con_umbral / por.juegos
guardarMetrica(por, "02_eda_juegos_por_anio")
display(por.T.round(1))

fig, ax = plt.subplots(figsize=(10, 3.8))
colores = [COLOR if a <= ANIO_MAX else "lightgray" for a in por.index]
ax.bar(por.index, por.juegos, color=colores, label="lanzamientos")
ax.bar(por.index, por.con_umbral, color=COLOR_2, label=f"con ≥{UMBRAL} reseñas")
ax.set(title="Figura 5. Lanzamientos por año (gris: excluidos por D-002) y los que llegan al umbral", xlabel="año de lanzamiento", ylabel="juegos")
ax.legend(); guardarFigura(fig, "02_eda_05_cohortes"); plt.show()

edad = (pd.Timestamp(config.VERSION_DATOS["fecha_max_lanzamiento"]) - m.fecha).dt.days / 365.25
m["edad_cat"] = pd.cut(edad.clip(upper=10), [0, .5, 1, 2, 4, 10.01], labels=["<6 m", "6-12 m", "1-2 a", "2-4 a", "4+ a"])
tab_edad = m.groupby("edad_cat").pct_pos.agg(["size", "mean", "std"])
display(tab_edad.round(3))
R["pct_umbral_2015"], R["pct_umbral_2024"] = por.loc[2015, "pct_con_umbral"], por.loc[2024, "pct_con_umbral"]
R["juegos_2024"] = int(por.loc[2024, "juegos"]); R["juegos_2015"] = int(por.loc[2015, "juegos"])
R["pct_4mas"] = 100 * (tab_edad.loc["4+ a", "size"] / tab_edad["size"].sum())

**Lectura.**
- La oferta crece cada año, pero la proporción de juegos que llega al umbral cae: la muestra describe **juegos que lograron cierta tracción**, no el catálogo completo (sesgo de selección que se declara como limitación).
- Excluir 2025 y 2026 (D-002) elimina casi todos los juegos de menos de 12 meses, por lo que la regla de antigüedad mínima de 12 meses propuesta en la sesión queda **cubierta por D-002**: no hace falta una segunda regla.
- La gran mayoría de la muestra tiene 4 o más años y su recepción media es menor; eso es un efecto de época (siguiente sección), no de reseñas «inmaduras».

## 4. Deriva temporal y justificación de la partición

In [ ]:
anual = m.groupby("anio").pct_pos.agg(media="mean", mediana="median", q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75), juegos="size")
anual = anual.loc[2010:]
guardarMetrica(anual, "02_eda_deriva_anual")
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.fill_between(anual.index, anual.q25, anual.q75, color=COLOR, alpha=.2, label="rango intercuartílico")
ax.plot(anual.index, anual["mediana"], marker="o", color=COLOR, label="mediana")
ax.plot(anual.index, anual["media"], marker="s", color=COLOR_2, label="media")
ax.set(title="Figura 6. La recepción por año de lanzamiento no es estacionaria", xlabel="año de lanzamiento", ylabel="pct_pos"); ax.legend()
guardarFigura(fig, "02_eda_06_deriva"); plt.show()
R["media_2014"], R["media_2024"] = anual.loc[2014, "media"], anual.loc[2024, "media"]

# Partición temporal propuesta: 70 / 15 / 15 por fecha de lanzamiento
c1, c2 = m.fecha.quantile(config.FRACCION_TRAIN), m.fecha.quantile(config.FRACCION_TRAIN + config.FRACCION_VALID)
m["conjunto"] = np.where(m.fecha <= c1, "train", np.where(m.fecha <= c2, "valid", "test"))
res = m.groupby("conjunto").agg(juegos=("pct_pos", "size"), desde=("fecha", "min"), hasta=("fecha", "max"), media=("pct_pos", "mean"), desv=("pct_pos", "std")).loc[["train", "valid", "test"]]
display(res)
guardarMetrica(res, "02_eda_particion_propuesta")

tr, te = m[m.conjunto == "train"], m[m.conjunto == "test"]
rmse = lambda y, p: float(np.sqrt(np.mean((y - p) ** 2)))
base_media = rmse(te.pct_pos, tr.pct_pos.mean())
base_cohorte = rmse(te.pct_pos, te.groupby("anio").pct_pos.transform("mean"))
print(f"Referencia ingenua (media del train) en test: RMSE = {base_media:.4f} · sesgo = {te.pct_pos.mean() - tr.pct_pos.mean():+.3f}")
print(f"Tasa base de la cohorte (oráculo, usa la media del propio año): RMSE = {base_cohorte:.4f}")
print(f"Desviación del test: {te.pct_pos.std():.4f}")
R.update(cut_train=c1.date(), cut_valid=c2.date(), rmse_media=base_media, rmse_cohorte=base_cohorte, sesgo_test=te.pct_pos.mean() - tr.pct_pos.mean(), n_train=len(tr), n_test=len(te))

**Lectura.** La media de `pct_pos` sube año con año; una partición aleatoria mezclaría épocas y sobreestimaría el desempeño, así que se usa **partición temporal por fecha de lanzamiento** (decisión 0003). Consecuencias para el modelado:
1. La referencia «predecir la media del train» queda sesgada a la baja en el test; ese sesgo debe reportarse.
2. La «tasa base de la cohorte» (hipótesis nula del planteamiento) es una referencia más exigente: mide cuánto aporta un modelo **más allá de saber en qué año salió el juego**. Como el año no puede ser variable (el test cae en años que el modelo no vio), la comparación justa es con la media del año anterior.
3. Se evaluará siempre sin pesos (D-008).

## 5. Predictores disponibles antes del lanzamiento

Todas las comparaciones usan la muestra (≥50 reseñas, hasta 2024). Además de `pct_pos`, se reporta `dif_cohorte` (diferencia contra la media de su año), que descuenta la tendencia temporal y evita confundir «época» con «característica».

### 5.1 Precio (valor actual de la tienda, Pre\*)

In [ ]:
u_ok["precio_pct_cohorte"] = u_ok.groupby("anio").price.rank(pct=True)
m["precio_pct_cohorte"] = u_ok["precio_pct_cohorte"]
m["log_precio"] = np.log1p(m.price)
m["precio_rango"] = pd.cut(m.price, [-0.01, 0, 5, 10, 20, 40, 1e9], labels=["gratis", "0-5", "5-10", "10-20", "20-40", "40+"])
tab_precio = m.groupby("precio_rango").agg(juegos=("pct_pos", "size"), media=("pct_pos", "mean"), dif_cohorte=("dif_cohorte", "mean"))
guardarMetrica(tab_precio, "02_eda_precio")
display(tab_precio.round(3))
print(f"Precio nulo (juegos retirados) en la muestra: {int(m.price.isna().sum())} · Spearman log_precio vs pct_pos: {rho(m.log_precio, m.pct_pos):.3f} · vs dif_cohorte: {rho(m.log_precio, m.dif_cohorte):.3f}")

pagados = m[m.price > 0].copy()
pagados["decil"] = pd.qcut(pagados.precio_pct_cohorte, 10, labels=False, duplicates="drop")
dec = pagados.groupby("decil").dif_cohorte.agg(["mean", "sem"])

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(m.log_precio.dropna(), bins=40, color=COLOR); ax[0].set(title="Figura 7a. log(1 + precio)", xlabel="log(1 + USD)")
sns.boxplot(data=m, x="precio_rango", y="pct_pos", color=COLOR, ax=ax[1], fliersize=1); ax[1].set(title="Figura 7b. pct_pos por rango de precio", xlabel="USD")
ax[2].errorbar(dec.index, dec["mean"], yerr=1.96 * dec["sem"], marker="o", color=COLOR, capsize=3)
ax[2].axhline(0, color="k", lw=.8); ax[2].set(title="Figura 7c. Diferencia vs. su cohorte según precio", xlabel="decil de precio dentro del año (juegos de pago)", ylabel="dif_cohorte")
guardarFigura(fig, "02_eda_07_precio"); plt.show()
R["rho_precio"] = rho(m.log_precio, m.dif_cohorte)
R["precio_gratis_dif"] = tab_precio.loc["gratis", "dif_cohorte"]

**Lectura.** La relación con el precio es **no lineal** (H3): los juegos de 5 a 20 USD se reciben mejor que su cohorte, los de menos de 5 y los gratuitos están alrededor o ligeramente debajo del promedio, y los de 40 USD o más quedan por debajo. Esto favorece modelos que capturen no linealidades (árboles) frente a uno lineal. Advertencia: es el precio actual, no el del lanzamiento (ver sección 8).

### 5.2 Plataformas y mes de lanzamiento (pregunta abierta de la sesión del 22/11)

In [ ]:
m["mac"] = m.mac.astype("string").eq("True")
m["linux"] = m.linux.astype("string").eq("True")
m["windows"] = m.windows.astype("string").eq("True")
filas = []
for p in ["mac", "linux"]:
    for v in [False, True]:
        s = m[m[p] == v]
        filas.append({"plataforma": p, "soportada": v, "juegos": len(s), "pct": 100 * len(s) / len(m), "media": s.pct_pos.mean(), "dif_cohorte": s.dif_cohorte.mean()})
tab_plat = pd.DataFrame(filas).set_index(["plataforma", "soportada"])
guardarMetrica(tab_plat, "02_eda_plataformas")
display(tab_plat.round(3))
print(f"Windows en la muestra: {m.windows.mean():.1%} (casi constante → se descarta como variable)")

m["mes"] = m.fecha.dt.month
tab_mes = m.groupby("mes").dif_cohorte.agg(["mean", "sem", "size"])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(["mac", "linux"], [m[m.mac].dif_cohorte.mean() - m[~m.mac].dif_cohorte.mean(), m[m.linux].dif_cohorte.mean() - m[~m.linux].dif_cohorte.mean()], color=COLOR)
ax[0].axhline(0, color="k", lw=.8); ax[0].set(title="Figura 8a. Diferencia en dif_cohorte (con − sin soporte)", ylabel="diferencia")
ax[1].errorbar(tab_mes.index, tab_mes["mean"], yerr=1.96 * tab_mes["sem"], marker="o", color=COLOR, capsize=3)
ax[1].axhline(0, color="k", lw=.8); ax[1].set(title="Figura 8b. dif_cohorte por mes de lanzamiento", xlabel="mes")
guardarFigura(fig, "02_eda_08_plataforma_mes"); plt.show()
R["mac_dif"] = tab_plat.loc[("mac", True), "dif_cohorte"] - tab_plat.loc[("mac", False), "dif_cohorte"]
R["linux_dif"] = tab_plat.loc[("linux", True), "dif_cohorte"] - tab_plat.loc[("linux", False), "dif_cohorte"]
R["mes_rango"] = tab_mes["mean"].max() - tab_mes["mean"].min()

**Lectura.** Casi todos los juegos salen en Windows, así que esa variable no discrimina. Los juegos con soporte para Mac o Linux se reciben mejor que su cohorte (unos 5 puntos porcentuales de diferencia y Spearman cercano a 0.15), así que la hipótesis H6 (la plataforma aporta poco) **no se sostiene**; probablemente son un proxy del esfuerzo técnico del estudio y no un efecto causal. El mes de lanzamiento muestra una variación pequeña; se conserva como variable cíclica de bajo costo, pero no se espera que pese.

### 5.3 Géneros y etiquetas de marketing (D-005)

In [ ]:
def tipoEtiqueta(g):
    if g in config.ETIQUETAS_MARKETING: return "marketing"
    if g in config.AVISO_CONTENIDO: return "aviso de contenido"
    if g in config.GENEROS_JUEGO: return "género"
    return "otro"

def resumenEtiquetas(d, col, minimo=100):
    e = d[[col, "pct_pos", "dif_cohorte"]].explode(col).dropna(subset=[col])
    r = e.groupby(col).agg(juegos=("pct_pos", "size"), media=("pct_pos", "mean"), dif_cohorte=("dif_cohorte", "mean"))
    return r.query("juegos >= @minimo").sort_values("juegos", ascending=False)

gen = resumenEtiquetas(m, "lista_gen")
gen["tipo"] = [tipoEtiqueta(g) for g in gen.index]
guardarMetrica(gen, "02_eda_generos")
display(gen.round(3))
print(f"Géneros distintos tras normalizar: {len(set(g for L in u.lista_gen for g in L))} · juegos sin género en la muestra: {int((m.lista_gen.map(len) == 0).sum())}")

g = gen.sort_values("dif_cohorte")
fig, ax = plt.subplots(figsize=(8, 5))
paleta = {"género": COLOR, "marketing": COLOR_2, "aviso de contenido": "#8c564b", "otro": "gray"}
ax.barh(g.index, g.dif_cohorte, color=[paleta[t] for t in g.tipo])
ax.axvline(0, color="k", lw=.8)
ax.set(title="Figura 9. Recepción vs. su cohorte por género / etiqueta (≥100 juegos)", xlabel="dif_cohorte medio")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in paleta.values()]
ax.legend(handles, paleta.keys(), loc="lower right", fontsize=8)
guardarFigura(fig, "02_eda_09_generos"); plt.show()
R["peor_genero"] = (g.index[0], g.dif_cohorte.iloc[0]); R["mejor_genero"] = (g.index[-1], g.dif_cohorte.iloc[-1])
R["n_generos"] = len(set(x for L in u.lista_gen for x in L))

**Lectura.** Las diferencias entre géneros se leen en la Figura 9 (positivas arriba de su cohorte, negativas abajo): *Massively Multiplayer* y *Early Access* son los más rezagados. *Early Access* y *Free To Play* se comportan distinto del tipo de juego, lo que respalda separar marketing de género (D-005, H2). Los juegos con más de un género comparten observaciones entre filas, por lo que los tamaños no deben sumarse.

### 5.4 Categorías de Steam (funciones del producto)

In [ ]:
cat = resumenEtiquetas(m, "lista_cat", minimo=300)
cat = cat.drop(index=[c for c in config.CATEGORIAS_EXCLUIR if c in cat.index], errors="ignore")
guardarMetrica(cat, "02_eda_categorias")
top_cat = cat.head(18).sort_values("dif_cohorte")
display(top_cat.round(3))
frec = pd.Series([x for L in u_ok.lista_cat for x in L]).value_counts()
print(f"Categorías distintas: {len(frec)} · con ≥1 % de los juegos: {int((frec >= 0.01 * len(u_ok)).sum())}")

fig, ax = plt.subplots(figsize=(8, 5.2))
ax.barh(top_cat.index, top_cat.dif_cohorte, color=np.where(top_cat.dif_cohorte >= 0, COLOR, COLOR_2))
ax.axvline(0, color="k", lw=.8); ax.set(title="Figura 10. Recepción vs. su cohorte por categoría (las 18 más frecuentes)", xlabel="dif_cohorte medio")
guardarFigura(fig, "02_eda_10_categorias"); plt.show()
R["cat_mejor"] = (top_cat.index[-1], top_cat.dif_cohorte.iloc[-1]); R["cat_peor"] = (top_cat.index[0], top_cat.dif_cohorte.iloc[0])

**Lectura.** Las funciones que señalan cuidado del producto (Steam Cloud, soporte de control, tablas de clasificación, Workshop) quedan por encima de su cohorte, y las categorías multijugador en línea (Online Co-op, Online PvP, PvP) por debajo (H5). *Steam Trading Cards* se excluye porque Steam la activa según las ventas, no el desarrollador.

### 5.5 Idiomas (acuerdo A-007)

`supported_languages` mezcla idiomas con variantes regionales (p. ej. *Portuguese – Brazil* y *Portuguese – Portugal*) y, en unos 28 mil juegos, viene como un solo texto HTML o separado por comas. Se resuelve con `parsers.idiomasYAudio` y se agrupa por idioma base (lo anterior a « - »). La conversión a «mercado potencial» (número de hablantes, propuesta de Filiberto) queda como extensión.

In [ ]:
raw_idi = raw["supported_languages"].dropna().astype(str)
print(f"Celdas con marcas HTML: {int(raw_idi.str.contains('<').sum()):,}")
base = lambda L: {i.split(" - ")[0].strip() for i in L}
u_ok["n_idi"] = u_ok.lista_idi.map(lambda L: len(base(L)))
u_ok["n_audio"] = u_ok.lista_audio.map(lambda L: np.nan if L is None else len(base(L)))
u_ok["idi_sin_dato"] = u_ok.lista_idi.map(len) == 0
m["n_idi"], m["n_audio"], m["idi_sin_dato"] = u_ok.loc[m.index, "n_idi"], u_ok.loc[m.index, "n_audio"], u_ok.loc[m.index, "idi_sin_dato"]
m["tiene_ingles"] = m.lista_idi.map(lambda L: "English" in L)
todos = pd.Series([i for L in u.lista_idi for i in base(L)]).value_counts()
print(f"Idiomas base distintos: {len(todos)} · los menos frecuentes: {todos.tail(5).to_dict()}")

m["idi_cat"] = pd.cut(m.n_idi.where(~m.idi_sin_dato), [0, 1, 2, 5, 10, 100], labels=["1", "2", "3-5", "6-10", "11+"])
tab_idi = m.groupby("idi_cat").agg(juegos=("pct_pos", "size"), media=("pct_pos", "mean"), dif_cohorte=("dif_cohorte", "mean"), ee=("dif_cohorte", "sem"))
guardarMetrica(tab_idi, "02_eda_idiomas")
display(tab_idi.round(3))
rho_idi, rho_idi_c = rho(m.n_idi.where(~m.idi_sin_dato), m.pct_pos), rho(m.n_idi.where(~m.idi_sin_dato), m.dif_cohorte)
rho_aud = rho(m.n_audio, m.dif_cohorte)
print(f"Spearman # idiomas vs pct_pos: {rho_idi:.3f} · vs dif_cohorte: {rho_idi_c:.3f} · # idiomas con audio vs dif_cohorte: {rho_aud:.3f}")
print(f"Sin dato de idiomas en la muestra: {int(m.idi_sin_dato.sum())} · con inglés: {m.tiene_ingles.mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(m.n_idi.clip(upper=30) + np.random.uniform(-.25, .25, len(m)), m.pct_pos, s=2, alpha=.15, color=COLOR)
med = m.groupby(m.n_idi.clip(upper=30)).pct_pos.mean()
ax[0].plot(med.index, med.values, color=COLOR_2, lw=2, label="media por # idiomas"); ax[0].legend()
ax[0].set(title="Figura 11a. # de idiomas vs pct_pos (≥30 agrupados)", xlabel="# idiomas", ylabel="pct_pos")
ax[1].errorbar(range(len(tab_idi)), tab_idi.dif_cohorte, yerr=1.96 * tab_idi.ee.values, marker="o", color=COLOR, capsize=3)
ax[1].set_xticks(range(len(tab_idi))); ax[1].set_xticklabels(tab_idi.index); ax[1].axhline(0, color="k", lw=.8)
ax[1].set(title="Figura 11b. dif_cohorte por # de idiomas", xlabel="# idiomas")
guardarFigura(fig, "02_eda_11_idiomas"); plt.show()
R.update(rho_idi=rho_idi, rho_idi_c=rho_idi_c, rho_aud=rho_aud)

**Lectura.** Una vez contados correctamente los idiomas y descontada la cohorte, la asociación con la recepción es **débil** (Spearman cercano a 0; H4 solo se sostiene de forma leve). Además, si existiera, podría deberse a que los juegos exitosos se traducen **después** del lanzamiento (ver sección 8), por lo que `idi_n_idiomas` se incluye con reserva.

### 5.6 Ficha de la tienda: logros, sitio y soporte

In [ ]:
m["tiene_logros"] = m.achievements.fillna(0) > 0
for c in ["has_website", "has_support_url", "has_support_email", "has_notes"]:
    m[c] = m[c].astype("boolean").fillna(False).astype(bool)
filas = []
for etiqueta, col in [("tiene logros", "tiene_logros"), ("sitio web", "has_website"), ("URL de soporte", "has_support_url"), ("correo de soporte", "has_support_email"), ("aviso de contenido adulto", "has_notes")]:
    for v in [False, True]:
        s = m[m[col] == v]
        filas.append({"variable": etiqueta, "valor": v, "juegos": len(s), "media": s.pct_pos.mean(), "dif_cohorte": s.dif_cohorte.mean()})
tab_ficha = pd.DataFrame(filas).set_index(["variable", "valor"])
guardarMetrica(tab_ficha, "02_eda_ficha")
display(tab_ficha.round(3))
R["logros_dif"] = tab_ficha.loc[("tiene logros", True), "dif_cohorte"] - tab_ficha.loc[("tiene logros", False), "dif_cohorte"]
R["adulto_dif"] = tab_ficha.loc[("aviso de contenido adulto", True), "dif_cohorte"] - tab_ficha.loc[("aviso de contenido adulto", False), "dif_cohorte"]

## 6. Historial de desarrolladores y publishers (variable distintiva del proyecto)

El nombre del desarrollador no sirve (94 mil valores, casi todos con un solo juego), pero su **historial** sí puede. Dos precauciones de las discusiones del equipo:

1. **Solo el pasado** (A-004): para un juego lanzado en la fecha F se usan juegos lanzados antes de F.
2. **Reseñas maduras:** las reseñas de un juego reciente se siguen acumulando después de su lanzamiento, y el `pct_pos` que vemos hoy incluye opiniones que no existían cuando salió el siguiente juego del mismo estudio. Por eso se cuentan solo juegos con al menos 12 meses de antigüedad respecto al juego actual.

El prototipo calcula las dos versiones (con y sin la regla de madurez) para medir cuánto cambia.

In [ ]:
def historialPrevio(d, col, meses=12, umbral=UMBRAL):
    """Prototipo del historial sin fuga (Diana lo industrializa en variablesDesarrollador / variablesPublisher).

    Para cada juego devuelve: n_prev (juegos de la entidad lanzados antes), n_cal (previos con >= `meses` de
    antigüedad y >= `umbral` reseñas) y pct_prev (media de su pct_pos). Con varias entidades se agregan todas.
    """
    e = d[["app_id", "fecha", "pct_pos", "n_rev", col]].explode(col).dropna(subset=[col, "fecha"])
    e = e.drop_duplicates(["app_id", col]).sort_values([col, "fecha"])
    e["elig"] = (e.n_rev >= umbral)
    e["val"] = np.where(e.elig, e.pct_pos.fillna(0), 0.0)
    multi = e[e.groupby(col).app_id.transform("size") > 1]
    lag = pd.DateOffset(months=meses)
    ids, prev, cal, suma = [], [], [], []
    for _, g in multi.groupby(col, sort=False):
        f = g.fecha.values
        cumE = np.concatenate([[0], np.cumsum(g.elig.values)])
        cumV = np.concatenate([[0], np.cumsum(g.val.values)])
        i_prev = np.searchsorted(f, f, side="left")
        lim = (g.fecha - lag).values
        i_mad = np.searchsorted(f, lim, side="right" if meses > 0 else "left")
        ids.append(g.app_id.values); prev.append(i_prev); cal.append(cumE[i_mad]); suma.append(cumV[i_mad])
    r = pd.DataFrame({"app_id": np.concatenate(ids), "n_prev": np.concatenate(prev), "n_cal": np.concatenate(cal), "suma": np.concatenate(suma)})
    r = r.groupby("app_id").agg(n_prev=("n_prev", "max"), n_cal=("n_cal", "sum"), suma=("suma", "sum"))
    r = r.reindex(d.app_id.values).fillna(0)
    r["pct_prev"] = (r.suma / r.n_cal).where(r.n_cal > 0)
    return r[["n_prev", "n_cal", "pct_prev"]].set_axis(d.index)


dev12 = historialPrevio(u, "lista_dev", meses=12)
dev0 = historialPrevio(u, "lista_dev", meses=0)
pub12 = historialPrevio(u, "lista_pub", meses=12)
for nombre, h in [("dev12", dev12), ("dev0", dev0), ("pub12", pub12)]:
    for c in h.columns:
        m[f"{nombre}_{c}"] = h.loc[m.index, c]

cob = pd.DataFrame({
    "con juegos previos": [(m.dev0_n_prev > 0).mean(), (m.dev12_n_prev > 0).mean(), (m.pub12_n_prev > 0).mean()],
    "con historial calificado": [(m.dev0_n_cal > 0).mean(), (m.dev12_n_cal > 0).mean(), (m.pub12_n_cal > 0).mean()],
    "Spearman vs pct_pos": [rho(m.dev0_pct_prev, m.pct_pos), rho(m.dev12_pct_prev, m.pct_pos), rho(m.pub12_pct_prev, m.pct_pos)],
    "Spearman vs dif_cohorte": [rho(m.dev0_pct_prev, m.dif_cohorte), rho(m.dev12_pct_prev, m.dif_cohorte), rho(m.pub12_pct_prev, m.dif_cohorte)],
}, index=["desarrollador, sin regla de madurez", "desarrollador, 12 meses de madurez", "publisher, 12 meses de madurez"])
guardarMetrica(cob, "02_eda_historial")
display(cob.round(3))
print(f"Desarrolladores distintos: {len(set(x for L in u.lista_dev for x in L)):,} · publishers: {len(set(x for L in u.lista_pub for x in L)):,}")
m["autopublicado"] = [bool(set(a) & set(b)) for a, b in zip(m.lista_dev, m.lista_pub)]
print(f"Autopublicados (algún desarrollador es también publisher): {m.autopublicado.mean():.1%} · dif_cohorte medio: auto {m[m.autopublicado].dif_cohorte.mean():+.3f} vs externo {m[~m.autopublicado].dif_cohorte.mean():+.3f}")

ok = m.dropna(subset=["dev12_pct_prev"]).copy()
ok["decil"] = pd.qcut(ok.dev12_pct_prev, 10, labels=False, duplicates="drop")
dd = ok.groupby("decil").pct_pos.agg(["mean", "std"])
m["prev_cat"] = pd.cut(m.dev12_n_prev, [-1, 0, 1, 4, 9, 10000], labels=["0", "1", "2-4", "5-9", "10+"])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.9))
ax[0].errorbar(dd.index, dd["mean"], yerr=dd["std"], marker="o", color=COLOR, capsize=3)
ax[0].set(title="Figura 12a. pct_pos vs. historial maduro del desarrollador", xlabel="decil de pct_pos previo (solo con historial)", ylabel="pct_pos medio ± 1 desv.")
sns.boxplot(data=m, x="prev_cat", y="dif_cohorte", color=COLOR, ax=ax[1], fliersize=1)
ax[1].set(title="Figura 12b. dif_cohorte según # de juegos previos", xlabel="# de juegos previos del desarrollador")
guardarFigura(fig, "02_eda_12_historial"); plt.show()
R.update(cov_dev=(m.dev12_n_cal > 0).mean(), prev_dev=(m.dev12_n_prev > 0).mean(), rho_dev0=cob.iloc[0, 2], rho_dev12=cob.iloc[1, 2], rho_dev12_c=cob.iloc[1, 3], rho_pub=cob.iloc[2, 2], auto_pct=m.autopublicado.mean())

**Lectura.** El historial de calidad es el predictor con mayor asociación (H1), pero la regla de madurez reduce la correlación y la cobertura frente a la versión ingenua: ese es el costo de evitar fuga. Quien nunca ha publicado (*cold start*) no tiene historial: se necesita un indicador `dev_sin_historial` y un valor de relleno, y el modelo debe funcionar también sin esta variable (se reportará con y sin historial). El historial del publisher y el del desarrollador están muy correlacionados (≈0.8). En la partición final el historial debe calcularse con la misma regla en train y test (solo pasado), y para descartar memorización por estudio se evaluará también con partición agrupada por desarrollador.

## 7. Auditoría de fuga de información (variables POST)

Columnas generadas después del lanzamiento pueden correlacionar con el objetivo precisamente porque son consecuencia del éxito. Se mide su correlación para **documentar** por qué se excluyen (no por su valor predictivo).

In [ ]:
post = m.copy()
filas = []
for c in ["recommendations", "peak_ccu", "metacritic_score", "user_score", "average_playtime_forever", "median_playtime_forever", "average_playtime_2weeks", "median_playtime_2weeks"]:
    s = pd.to_numeric(post[c], errors="coerce").replace(0, np.nan)
    filas.append({"variable": c, "pct_con_dato": 100 * s.notna().mean(), "spearman_vs_pct_pos": rho(s, post.pct_pos)})
dueños = post.estimated_owners.astype("string").str.extract(r"^\s*([\d,]+)")[0].str.replace(",", "").astype(float).replace(0, np.nan)
filas.append({"variable": "estimated_owners (límite inferior del rango)", "pct_con_dato": 100 * dueños.notna().mean(), "spearman_vs_pct_pos": rho(dueños, post.pct_pos)})
fuga = pd.DataFrame(filas).set_index("variable").sort_values("spearman_vs_pct_pos", key=abs, ascending=False)
guardarMetrica(fuga, "02_eda_fuga_post")
display(fuga.round(3))
fig, ax = plt.subplots(figsize=(8, 3.8))
f = fuga.iloc[::-1]
ax.barh(f.index, f.spearman_vs_pct_pos, color=COLOR_2); ax.axvline(0, color="k", lw=.8)
ax.set(title="Figura 13. Correlación de variables POST con pct_pos (se EXCLUYEN del modelo)", xlabel="Spearman")
guardarFigura(fig, "02_eda_13_fuga_post"); plt.show()
R["fuga_max"] = fuga[fuga.pct_con_dato >= 10].spearman_vs_pct_pos.abs().max()

**Decisión.** Todas las variables POST (reseñas, propietarios, jugadores simultáneos, tiempos de juego, Metacritic, recomendaciones, puntuación de usuarios) y las etiquetas de usuarios (`tags`) quedan **excluidas como predictoras**; `positive` y `negative` solo sirven para construir el objetivo y filtrar. Correlacionan con el objetivo, y esa es justo la señal de que dependen de él (`user_score` casi lo reproduce, pero solo existe para unos 70 juegos). `metacritic_score` solo existe para títulos con crítica profesional y es posterior al lanzamiento: no se usa ni siquiera como indicador `has_*`.

## 8. Variables «Pre\*»: el valor guardado es el de hoy, no el del lanzamiento

Precio, idiomas, categorías, capturas y tráileres reflejan el estado de la tienda el 24 de noviembre de 2026. El riesgo no es solo ruido: los juegos que tuvieron éxito reciben **después** traducciones, rebajas y contenido, así que parte de la señal podría ser *consecuencia* del éxito. No se puede corregir con estos datos, pero sí **medir**: si los juegos viejos tienen sistemáticamente más idiomas o categorías que los recientes, hay acumulación posterior al lanzamiento.

In [ ]:
a = u_ok[u_ok.anio.between(2012, ANIO_MAX)].copy()
a["n_cat"] = a.lista_cat.map(lambda L: len([c for c in L if c not in config.CATEGORIAS_EXCLUIR]))
a["tiene_logros"] = a.achievements.fillna(0) > 0
a["es_gratis"] = a.price == 0
a["sitio"] = a.has_website.astype("boolean").fillna(False).astype(bool)
a["correo"] = a.has_support_email.astype("boolean").fillna(False).astype(bool)
sd = ~a.idi_sin_dato
aud = a.groupby("anio").agg(
    precio_mediano=("price", "median"), pct_gratis=("es_gratis", lambda s: 100 * s.mean()),
    n_categorias=("n_cat", "mean"), pct_con_logros=("tiene_logros", lambda s: 100 * s.mean()),
    pct_sitio_web=("sitio", lambda s: 100 * s.mean()), pct_correo_soporte=("correo", lambda s: 100 * s.mean()))
aud["n_idiomas"] = a[sd].groupby("anio").n_idi.mean()
aud["n_idiomas_audio"] = a.groupby("anio").n_audio.mean()
guardarMetrica(aud, "02_eda_auditoria_pre_asterisco")
display(aud.round(2).T)

vars_ = ["precio_mediano", "pct_gratis", "n_idiomas", "n_idiomas_audio", "n_categorias", "pct_sitio_web"]
fig, ax = plt.subplots(2, 3, figsize=(13, 6))
for axx, v in zip(ax.ravel(), vars_):
    axx.plot(aud.index, aud[v], marker="o", color=COLOR); axx.set(title=v, xlabel="año de lanzamiento")
fig.suptitle("Figura 14. Variables Pre* por año de lanzamiento (todos los juegos, no solo la muestra)", y=1.02)
guardarFigura(fig, "02_eda_14_auditoria_pre_asterisco"); plt.show()
for v_ in vars_:
    print(f"{v_}: {aud[v_].iloc[0]:.2f} (2012) → {aud[v_].iloc[-1]:.2f} (2024)")
R.update(idi_2012=aud.loc[2012, "n_idiomas"], idi_2024=aud.loc[2024, "n_idiomas"], cat_2012=aud.loc[2012, "n_categorias"], cat_2024=aud.loc[2024, "n_categorias"],
         precio_2014=aud.loc[2014, "precio_mediano"], precio_2024=aud.loc[2024, "precio_mediano"])

**Lectura.** Resultado: todas las variables Pre* cambian mucho con el año de lanzamiento. Las categorías, los logros y el sitio web son **más frecuentes en los juegos viejos** (posible acumulación posterior al lanzamiento, o supervivencia de los títulos que siguen en la tienda); el precio mediano baja y los juegos gratuitos suben de 7 % a 26 %; y los idiomas **aumentan en los juegos recientes** (el catálogo moderno nace más localizado). En ambos casos el efecto es un confusor de época, que se controla (parcialmente) comparando contra la cohorte, y la forma defendible de valorar el riesgo es la **ablación** del Sprint 2 (entrenar con y sin las variables Pre\*).

Mitigaciones ya acordadas para el pipeline: percentil de precio dentro de la cohorte (`precio_pct_cohorte`), exclusión de `dlc_count` y de `tags`, y reporte explícito de la limitación.

## 9. Relaciones entre predictores

In [ ]:
cand = pd.DataFrame({
    "log_precio": m.log_precio, "precio_pct_cohorte": m.precio_pct_cohorte, "es_gratis": (m.price == 0).astype(int),
    "n_idiomas": m.n_idi.where(~m.idi_sin_dato), "n_audio": m.n_audio,
    "n_generos": m.lista_gen.map(len), "n_categorias": m.lista_cat.map(len),
    "mac": m.mac.astype(int), "linux": m.linux.astype(int), "tiene_logros": m.tiene_logros.astype(int),
    "sitio_web": m.has_website.astype(int), "mes": m.mes,
    "dev_n_previos": m.dev12_n_prev, "dev_pct_pos_previo": m.dev12_pct_prev, "pub_pct_pos_previo": m.pub12_pct_prev,
})
cm = cand.corr("spearman")
con_y = cand.apply(lambda s: rho(s, m.dif_cohorte)).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(15, 5.6), gridspec_kw={"width_ratios": [1.5, 1]})
sns.heatmap(cm, annot=True, fmt=".2f", cmap="coolwarm", center=0, cbar=False, ax=ax[0], annot_kws={"size": 7})
ax[0].set_title("Figura 15a. Correlación (Spearman) entre predictores")
ax[1].barh(con_y.index, con_y.values, color=np.where(con_y >= 0, COLOR, COLOR_2)); ax[1].axvline(0, color="k", lw=.8)
ax[1].set_title("Figura 15b. Correlación de cada predictor con dif_cohorte")
guardarFigura(fig, "02_eda_15_correlaciones"); plt.show()
guardarMetrica(con_y.to_frame("spearman_vs_dif_cohorte"), "02_eda_correlacion_predictores")
altas = [(a_, b_, round(float(cm.loc[a_, b_]), 2)) for i, a_ in enumerate(cm.columns) for b_ in cm.columns[i + 1:] if abs(cm.loc[a_, b_]) > .6]
print("Pares muy correlacionados (|rho| > 0.6):", altas)
print("Spearman con dif_cohorte:", con_y.round(3).to_dict())

## 10. Hallazgos y decisiones

In [ ]:
g_p, g_m = R["peor_genero"], R["mejor_genero"]
tabla = f"""
| # | Hallazgo | Evidencia | Decisión para el pipeline |
|---|---|---|---|
| 1 | Cada EDA usó una descarga distinta | 145,257 a 151,023 filas; 62,178 a 67,929 juegos sin reseñas | Una versión congelada ({R['n_raw']:,} filas, 24-nov-2026), verificada con `verificarVersion` |
| 2 | Filas a eliminar | {R['n_fechas']} fechas inválidas, 2 precios de 107,500 USD (ambos de 2026), {R['n_nojuego']:,} no-juegos | `limpiarFechas`, `limpiarPrecios`, `quitarNoJuegos` |
| 3 | Muy pocos juegos tienen reseñas | {R['sin_resenas_pct']:.0f} % del universo hasta 2024 sin reseñas; muestra de {R['n_m']:,} con ≥{UMBRAL} | Umbral {UMBRAL} (sensibilidad 30 y 100); declarar el sesgo de selección |
| 4 | El umbral reduce el ruido | Desv. de pct_pos: {R['std_t1']:.3f} (≥1) → {R['std_t50']:.3f} (≥50) | `UMBRAL_RESENAS = 50` |
| 5 | Objetivo asimétrico | Media {R['media']:.3f}, mediana {R['mediana']:.3f}, asimetría {R['asim']:.2f} | Reportar RMSE y MAE; objetivo `y_pct_pos`, secundario `y_bueno` |
| 6 | Wilson ≈ proporción cruda con ≥50 reseñas | Spearman {R['rho_wilson']:.3f} | No se usa Wilson como objetivo principal |
| 7 | Deriva temporal de la recepción | Media {R['media_2014']:.3f} (2014) → {R['media_2024']:.3f} (2024); sesgo del baseline en test {R['sesgo_test']:+.3f} | Partición temporal; cortes {R['cut_train']} y {R['cut_valid']}; año y antigüedad **no** son variables |
| 8 | El formato de `genres`/`categories` es un artefacto | Diferencia global {R['fmt_dif_global']:+.3f}, dentro del año {R['fmt_dif_cohorte']:+.3f} | `parsers.aLista`; el formato no es variable |
| 9 | D-002 cubre la antigüedad mínima | 2025–2026 concentran los juegos de menos de 12 meses | No se agrega una regla de antigüedad adicional |
| 10 | Precio: relación no lineal con la recepción | Spearman log-precio vs dif_cohorte {R['rho_precio']:+.3f}; gratis {R['precio_gratis_dif']:+.3f} vs su cohorte | `precio_log`, `precio_es_gratis`, `precio_rango`, `precio_pct_cohorte` |
| 11 | Mac y Linux se asocian con mejor recepción (posible proxy del esfuerzo del estudio) | Mac {R['mac_dif']:+.3f}, Linux {R['linux_dif']:+.3f} (dif_cohorte, con − sin soporte) | Conservar mac y linux; descartar windows (constante) |
| 12 | Los géneros se distinguen moderadamente | {g_p[0]} {g_p[1]:+.3f}; {g_m[0]} {g_m[1]:+.3f} | Multi-hot; marketing separado (D-005) |
| 13 | Las funciones del producto se asocian con la recepción | Mejor: {R['cat_mejor'][0]} {R['cat_mejor'][1]:+.3f}; peor: {R['cat_peor'][0]} {R['cat_peor'][1]:+.3f} | Multi-hot de categorías con ≥1 %; excluir Trading Cards |
| 14 | Idiomas: asociación débil con la recepción | Spearman {R['rho_idi_c']:+.3f} vs dif_cohorte | `idi_n_idiomas`, `idi_n_audio` con reserva (Pre\\*) |
| 15 | El historial del desarrollador es la señal más fuerte | Spearman {R['rho_dev12']:.3f} con regla de madurez ({R['rho_dev0']:.3f} sin ella); cobertura {R['cov_dev']:.0%} | `dev_pct_pos_previo` con 12 meses de madurez + indicador de cold start; reportar con y sin historial |
| 16 | Variables POST correlacionan con el objetivo | Spearman máximo {R['fuga_max']:.2f} entre las variables con ≥10 % de cobertura | Excluir todas; solo `positive`/`negative` construyen el objetivo |
| 17 | Las variables Pre* cambian con la época y pueden acumular señal posterior | {R['idi_2012']:.1f} idiomas medios en 2012 vs {R['idi_2024']:.1f} en 2024; categorías {R['cat_2012']:.1f} vs {R['cat_2024']:.1f} | Auditoría por año, ablación con/sin Pre* en el Sprint 2, limitación declarada |
"""
display(Markdown(tabla))
(RAIZ / "reports" / "metrics").mkdir(parents=True, exist_ok=True)
(RAIZ / "reports" / "metrics" / "02_eda_hallazgos.md").write_text(tabla, encoding="utf-8")
print({k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in R.items()})

## 11. Pendientes que este EDA deja a los siguientes pasos

- **Decisión 0001 (objetivo):** el equipo elige entre `y_pct_pos` (propuesto como principal) y la etiqueta de cuartil superior de la cohorte.
- **Decisión 0002 (umbral):** confirmar 50 y revisar la sensibilidad a 30 y 100 con el pipeline (`sensibilidadMuestra`).
- **Decisión 0003 (partición):** confirmar cortes por fecha; considerar también una partición agrupada por desarrollador.
- **D-010:** formalizar el tratamiento de las variables Pre\* y planear la ablación del Sprint 2.
- Auditar con la tienda una muestra de juegos 2021–2024 con cero reseñas (¿ceros reales o datos sin actualizar?).
- Idiomas como «mercado potencial» (número de hablantes): extensión opcional.
- No se evaluaron `tags` de usuarios (excluidos del archivo por fuga) ni texto de descripciones (D-007).

*Todas las figuras se guardaron en `reports/figures/02_eda_*.png` y las tablas en `reports/metrics/02_eda_*.csv`.*